# Main file to compare models on eGFR dataset 

In [3]:
import os
import copy
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.svm import SVC
from kan import KAN, ex_round
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.model_selection import StratifiedKFold, KFold

import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score, roc_curve,
    r2_score, mean_absolute_error, mean_squared_error
)

fold_path = "/home/alecacciatore/ECML26/GNN4eGFR"
out_path = os.path.join(fold_path, "results_eGFR")
file_path = os.path.join(fold_path, "XY_temp.csv")
file_updated_path = os.path.join(fold_path, "XY_temp_updated.csv")
file_no_temp_path = os.path.join(fold_path, "XY_no_temp_updated.csv")

if not os.path.exists(out_path):
    os.makedirs(out_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Get data

In [4]:
# read the CSV file into a DataFrame
df = pd.read_csv(file_no_temp_path)

# remove icd9 columns
cols_ok = [col for col in df.columns[:-2] if not 'icd9' in col]

# the last column is the target variable and the first one is an indppex
y_classif = df.iloc[:, -1]
y_regress = df.iloc[:, -2]

# X columns
# X_no_icd9 = df[cols_ok]
# print(X_no_icd9.shape, f"with {X_no_icd9.isnull().sum().sum()} missing values")
X = df.iloc[:, 1:-3]
print(X.shape, f"with {X.isnull().sum().sum()} missing values")
X = X.fillna(X.mean())
print(X.shape, f"with {X.isnull().sum().sum()} missing values")

# y_classif contains [I, II, IIIa, IIIb, IV, V] labels
label_mapping = {'I': 0, 'II': 1, 'IIIa': 2, 'IIIb': 3, 'IV': 4, 'V': 5}
# label_mapping = {'I': 0, 'II': 1, 'IIIa': 1, 'IIIb': 1, 'IV': 1, 'V': 1}
y_classif = y_classif.map(label_mapping)

# Split data into training and testing sets
X_train, X_test, y_classif_train, y_classif_test = train_test_split(
    X, y_classif, test_size=0.2, random_state=42, stratify=y_classif
)
X_train_reg, X_test_reg, y_regress_train, y_regress_test = train_test_split(
    X, y_regress, test_size=0.2, random_state=42
)

# Create a dataset dictionary for KAN ('test_input', 'test_label', 'train_input', 'train_label')
train_data_classif = {
    'train_input': torch.tensor(X_train.values, dtype=torch.float32).to(device),
    'train_label': torch.tensor(y_classif_train.values, dtype=torch.long).to(device),
    'test_input': torch.tensor(X_test.values, dtype=torch.float32).to(device),
    'test_label': torch.tensor(y_classif_test.values, dtype=torch.long).to(device)
}
train_data_regress = {
    'train_input': torch.tensor(X_train_reg.values, dtype=torch.float32).to(device),
    'train_label': torch.tensor(y_regress_train.values, dtype=torch.float32).to(device),
    'test_input': torch.tensor(X_test_reg.values, dtype=torch.float32).to(device),
    'test_label': torch.tensor(y_regress_test.values, dtype=torch.float32).to(device)
}

(1833, 539) with 2236 missing values
(1833, 539) with 0 missing values


## Imbalance ratio

In [5]:
class_counts = y_classif_train.value_counts().sort_index()
imbalance_ratio = class_counts.max() / class_counts.min()
print("Class distribution in training set:")
for cls, count in class_counts.items():
    print(f"Class {cls}: {count} samples")
print(f"Imbalance Ratio: {imbalance_ratio:.2f}")

Class distribution in training set:
Class 0: 264 samples
Class 1: 788 samples
Class 2: 250 samples
Class 3: 115 samples
Class 4: 42 samples
Class 5: 7 samples
Imbalance Ratio: 112.57


# Define models to be used

In [6]:
# KAN
kan_classifier33 = KAN(width=[train_data_classif['train_input'].shape[1], 2, 2], grid=3, k=3, device=device, ckpt_path=out_path)
kan_classifier55 = KAN(width=[train_data_classif['train_input'].shape[1], 2, 2], grid=5, k=5, device=device, ckpt_path=out_path)

# MLP
mlp_classifier = MLPClassifier(
    hidden_layer_sizes=(4,),      # leggermente più grande per equità
    activation='relu',
    solver='adam',
    alpha=1e-3,                   # regolarizzazione L2
    batch_size='auto',
    learning_rate_init=1e-3,
    max_iter=500,
    random_state=42
)

# SVB (Linear)
svm_linear = SVC(
    kernel='linear',
    C=1.0,
    probability=True,
    random_state=42
)

# SVB (RBF)
svm_rbf = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    probability=True,
    random_state=42
)

# XGBoost
xgb_classifier = XGBClassifier(
    n_estimators=100,        # moderato
    max_depth=3,             # shallow trees
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    use_label_encoder=False,
    eval_metric='logloss' if len(class_counts) == 2 else 'mlogloss',
    random_state=42
)

# EBM (Explainable Boosting Machine)
ebm_classifier = ExplainableBoostingClassifier(
    interactions=0,     # solo additive → fairness con KAN shallow
    max_bins=32,
    max_interaction_bins=16,
    learning_rate=0.01,
    random_state=42
) # TODO: provare con interactions=10 (più competitivo ma diventa più potente della KAN)

# Logistic Regression
logreg_classifier = LogisticRegression(
    penalty='l2',
    C=1.0,
    solver='lbfgs',
    max_iter=1000,
    random_state=42
)

checkpoint directory created: /home/alecacciatore/ECML26/GNN4eGFR/results_eGFR
saving model version 0.0
checkpoint directory created: /home/alecacciatore/ECML26/GNN4eGFR/results_eGFR
saving model version 0.0


# Define evaluation metrics

In [7]:
def compute_classification_metrics(y_true, y_pred, y_proba):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, average='binary' if len(class_counts) == 2 else 'weighted'),
        "auc": roc_auc_score(y_true, y_proba, multi_class='ovr'),
    }


def compute_regression_metrics(y_true, y_pred):
    return {
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred)
    }


def bootstrap_ci(metric_fn, y_true, y_pred, n_bootstrap=1000, alpha=0.95):
    scores = []
    n = len(y_true)
    for _ in range(n_bootstrap):
        idx = np.random.choice(n, n, replace=True)
        scores.append(metric_fn(y_true[idx], y_pred[idx]))

    lower = np.percentile(scores, (1-alpha)/2*100)
    upper = np.percentile(scores, (1+alpha)/2*100)
    return lower, upper

# Plot functions

In [ ]:
def plot_and_save(fig, save_path):
    fig.savefig(save_path)
    plt.close(fig)


def plot_roc(y_true, y_proba, model_name, out_dir, class_id=None):
    if class_id is None:
        fpr, tpr, _ = roc_curve(y_true, y_proba)
    else:
        fpr, tpr = y_true, y_proba
    fig = plt.figure()
    plt.plot(fpr, tpr)
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.title(f"ROC - {model_name} (class #{class_id})")
    plot_and_save(fig, os.path.join(out_dir, f"{model_name}_roc_{class_id}class.png"))


def plot_calibration(y_true, y_proba, model_name, out_dir):
    if len(class_counts) == 2:
        prob_true, prob_pred = calibration_curve(y_true, y_proba, n_bins=10)
        prob_true = {prob_true}
        prob_pred = {prob_pred}
    else:
        from sklearn.preprocessing import label_binarize
        y_test_bin = label_binarize(y_true, classes=range(len(class_counts)))
        
        prob_true = dict()
        prob_pred = dict()

        for i in range(len(class_counts)):
            prob_true[i], prob_pred[i] = calibration_curve(
                y_test_bin[:, i],
                y_proba[:, i],
                n_bins=10
            )
    
    for i in range(len(prob_true)):
        fig = plt.figure()
        plt.plot(prob_pred[i], prob_true[i])
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.title(f"Calibration - {model_name}" + f"class #{i}" if len(prob_true) > 2 else '')
        plot_and_save(fig, os.path.join(out_dir, f"{model_name}_calibration" +
                                         f"_{i}class" if len(prob_true) > 2 else '' + ".png"))

# Training functions

## Experiment runners

In [9]:
def train_eval_sklearn(model, X_train, y_train, X_test, y_test, task):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if task == "classification":
        y_proba = model.predict_proba(X_test) 
        metrics = compute_classification_metrics(y_test, y_pred, y_proba)
        probs = y_proba[:, 1] if len(class_counts) == 2 else y_proba
        return metrics, y_pred, probs
    else:
        metrics = compute_regression_metrics(y_test, y_pred)
        return metrics, y_pred, None

In [10]:
def train_eval_kan(kan_model, train_data, task, steps=300, lr=1e-3):
    optimizer = torch.optim.Adam(kan_model.parameters(), lr=lr)

    X_train = train_data["train_input"]
    y_train = train_data["train_label"]
    X_test = train_data["test_input"]
    y_test = train_data["test_label"]

    kan_model.train()

    for _ in range(steps):
        optimizer.zero_grad()
        outputs = kan_model(X_train)

        if task == "classification":
            loss = torch.nn.functional.cross_entropy(outputs, y_train)
        else:
            loss = torch.nn.functional.mse_loss(outputs.squeeze(), y_train)

        loss.backward()
        optimizer.step()

    kan_model.eval()
    with torch.no_grad():
        outputs = kan_model(X_test)

        if task == "classification":
            probs = torch.softmax(outputs, dim=1)[:, 1]
            y_pred = torch.argmax(outputs, dim=1)
            y_pred_np = y_pred.cpu().numpy()
            y_proba_np = probs.cpu().numpy()
            y_test_np = y_test.cpu().numpy()

            metrics = compute_classification_metrics(
                y_test_np, y_pred_np, y_proba_np
            )
            return metrics, y_pred_np, y_proba_np
        else:
            y_pred = outputs.squeeze()
            y_pred_np = y_pred.cpu().numpy()
            y_test_np = y_test.cpu().numpy()
            metrics = compute_regression_metrics(y_test_np, y_pred_np)
            return metrics, y_pred_np, None
        

# def train_eval_kan(kan_model, train_data, task, steps=300, lr=1e-3):

#     def train_acc():
#         return torch.mean((torch.argmax(kan_model(train_data['train_input']), dim=1) == train_data['train_label']).type(torch.float32))

#     def test_acc():
#         return torch.mean((torch.argmax(kan_model(train_data['test_input']), dim=1) == train_data['test_label']).type(torch.float32))

#     results = kan_model.fit(train_data, opt="LBFGS", steps=steps, metrics=(train_acc, test_acc),
#                                 loss_fn=torch.nn.CrossEntropyLoss(), )
#     print("Training completed."
#         f"\nFinal Train Accuracy: {results['train_acc'][-1]:.4f}"
#         f"\nFinal Test Accuracy: {results['test_acc'][-1]:.4f}")
    
#     with torch.no_grad():
#         outputs = kan_model(train_data['test_input'])

#     probs = torch.softmax(outputs, dim=1)[:, 1]
#     y_pred = torch.argmax(outputs, dim=1)
#     y_pred_np = y_pred.cpu().numpy()
#     y_proba_np = probs.cpu().numpy()
#     y_test_np = train_data['test_label'].cpu().numpy()

#     metrics = compute_classification_metrics(
#         y_test_np, y_pred_np, y_proba_np
#     )
#     return metrics, y_pred_np, y_proba_np

In [17]:
class ExperimentRunner:

    def __init__(self, output_dir="results"):
        self.output_dir = output_dir + "_binary" if len(class_counts) == 2 else output_dir + f"_{len(class_counts)}class"
        os.makedirs(self.output_dir, exist_ok=True)
        self.results = {}

    def run_sklearn_model(self, name, model, X_train, y_train, X_test, y_test, task):
        metrics, y_pred, y_proba = train_eval_sklearn(
            model, X_train, y_train, X_test, y_test, task
        )

        if task == "classification":
            if len(class_counts) == 2:
                plot_roc(y_test, y_proba, name, self.output_dir)
            else:
                from sklearn.preprocessing import label_binarize
                y_test_bin = label_binarize(y_test, classes=range(len(class_counts)))
                y_proba = model.predict_proba(X_test)
                fpr = dict()
                tpr = dict()

                for i in range(len(class_counts)):
                    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
                    plot_roc(fpr[i], tpr[i], name, self.output_dir, i)
            plot_calibration(y_test, y_proba, name, self.output_dir)

        self.results[name] = metrics

    def run_kan_model(self, name, kan_model, train_data, task, steps):
        metrics, y_pred, y_proba = train_eval_kan(
            kan_model, train_data, task, steps
        )

        y_test = train_data["test_label"].cpu().numpy()

        if task == "classification":
            plot_roc(y_test, y_proba, name, self.output_dir)
            plot_calibration(y_test, y_proba, name, self.output_dir)

        self.results[name] = metrics

    def summary(self):
        df = pd.DataFrame(self.results).T
        df.to_csv(os.path.join(self.output_dir, "summary.csv"))
        return df

## Cross-validation

In [18]:
def cross_validate(model_fn, X, y, task, n_splits=5):
    if task == "classification":
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    else:
        cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    results = []

    for train_idx, test_idx in cv.split(X, y):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        metrics, _, _ = model_fn(X_tr, y_tr, X_te, y_te)
        results.append(metrics)

    return pd.DataFrame(results).mean(), pd.DataFrame(results).std()

# Classification: training

In [19]:
runner_classif = ExperimentRunner(out_path)

print("\n\tTraining LogReg")
runner_classif.run_sklearn_model(
    "LogReg", logreg_classifier,
    X_train, y_classif_train,
    X_test, y_classif_test,
    task="classification"
)

# print("\tTraining SVM linear")
# runner_classif.run_sklearn_model(
#     "SVM_linear", svm_linear,
#     X_train, y_classif_train,
#     X_test, y_classif_test,
#     task="classification"
# )

# print("\tTraining SVM rbf")
# runner_classif.run_sklearn_model(
#     "SVM_rbf", svm_rbf,
#     X_train, y_classif_train,
#     X_test, y_classif_test,
#     task="classification"
# )

print("\n\tTraining XGB")
runner_classif.run_sklearn_model(
    "XGBoost", xgb_classifier,
    X_train, y_classif_train,
    X_test, y_classif_test,
    task="classification"
)

print("\n\tTraining EBM")
runner_classif.run_sklearn_model(
    "EBM", ebm_classifier,
    X_train, y_classif_train,
    X_test, y_classif_test,
    task="classification"
)

print("\n\tTraining MLP")
runner_classif.run_sklearn_model(
    "MLP", mlp_classifier,
    X_train, y_classif_train,
    X_test, y_classif_test,
    task="classification"
)

print("\n\tTraining KAN")
runner_classif.run_kan_model(
    "KAN 3x3", kan_classifier33,
    train_data_classif,
    task="classification",
    steps=20
)

runner_classif.run_kan_model(
    "KAN 5x5", kan_classifier55,
    train_data_classif,
    task="classification",
    steps=20
)

print(runner_classif.summary())


	Training LogReg


/home/alecacciatore/ECML26/venv_ecml26/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/alecacciatore/ECML26/venv_ecml26/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression



	Training XGB


/home/alecacciatore/ECML26/venv_ecml26/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [23:19:53] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



	Training EBM

	Training MLP

	Training KAN


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


# Regression: training

In [ ]:
# runner_reg = ExperimentRunner(out_path)

# runner_reg.run_sklearn_model(
#     "XGB_reg",
#     xgb_regressor,
#     X_train_reg, y_regress_train,
#     X_test_reg, y_regress_test,
#     task="regression"
# )

# runner_reg.run_sklearn_model(
#     "MLP_reg",
#     mlp_regressor,
#     X_train_reg, y_regress_train,
#     X_test_reg, y_regress_test,
#     task="regression"
# )

# runner_reg.run_sklearn_model(
#     "SVR",
#     svr_model,
#     X_train_reg, y_regress_train,
#     X_test_reg, y_regress_test,
#     task="regression"
# )

# runner_reg.run_kan_model(
#     "KAN_reg",
#     kan_regressor,
#     train_data_regress,
#     task="regression"
# )

# print(runner_reg.summary())

: 

: 

: 

: 